In [3]:
import polars as pl

# 1. 데이터 경로 설정
input_path = "/Users/hajiyoon/workspace/data/processed/최종_상품단위_데이터셋.csv"
output_path = "pareto_processed_results_v3.csv"

# 2. 데이터 로딩 (LazyFrame 사용)
lf = pl.scan_csv(input_path)

# 3. 데이터 전처리 및 필터링
df = lf.select([
    pl.col("상품중분류명").alias("중분류명"),
    pl.col("상품명"),
    pl.col("총_매출금액").alias("총매출금액")
]).filter(
    pl.col("중분류명") != "생수"  # '생수' 중분류 삭제
).collect()

# ① 중분류별 총 매출액 계산 및 병합
df = df.with_columns([
    pl.col("총매출금액").sum().over("중분류명").alias("중분류총매출금액")
])

# ② 중분류 내 내림차순 정렬 및 누적 비율 계산
df = df.sort(["중분류명", "총매출금액"], descending=[False, True])

df = df.with_columns([
    (pl.col("총매출금액").cum_sum().over("중분류명") / pl.col("중분류총매출금액")).alias("누적비율"),
    (pl.col("총매출금액") / pl.col("중분류총매출금액")).alias("중분류총매출금액중비율")
])

# ③ 파레토(80%) 필터링
df_filtered = df.filter(pl.col("누적비율") < 0.8)

# ④ PB 상품 기준 중분류 추출
pb_categories = (
    df_filtered.filter(pl.col("상품명").str.contains("PB"))
    .select("중분류명")
    .unique()
    .to_series()
    .to_list()
)

# ⑤ 특별 구제(예외) 조건 딕셔너리 설정
exception_mapping = {
    '기타전통주': '롯데)', '삼각김밥': '롯데)', '떡': '유라가)',
    '햄버거': '롯데)', '냉장피자': '안유성)', '호빵': 'KBO)',
    '국산맥주': '신동엽)', '양주': '신동엽)', '샌드위치': '롯데)',
    '김밥': '롯데)', '와인': '지비)', '초밥류': '롯데)',
    '도시락': '롯데)', '하이볼': '세븐브로이)', '고급': 'G)',
    '냉동디저트': '시원)'
}

# 예외 조건에 맞는 중분류 찾기
exception_categories = []
for cat, keyword in exception_mapping.items():
    # 해당 중분류명이고, 상품명에 키워드가 있는 상품이 1개라도 있는지 확인 (literal=True로 괄호 문자 그대로 인식)
    match_count = df_filtered.filter(
        (pl.col("중분류명") == cat) & (pl.col("상품명").str.contains(keyword, literal=True))
    ).height
    
    if match_count > 0:
        exception_categories.append(cat)

# 살려둘 최종 중분류 목록 (PB 포함 + 예외 조건 포함, 중복 제거)
categories_to_keep = list(set(pb_categories + exception_categories))

# 최종 결과 필터링
final_df = df_filtered.filter(pl.col("중분류명").is_in(categories_to_keep))

# 4. 결과 CSV 저장
final_columns = ['중분류명', '상품명', '총매출금액', '중분류총매출금액중비율', '중분류총매출금액']
final_df.select(final_columns).write_csv(output_path)

# 5. 마지막 출력 (PB 유무 + 구제된 중분류 확인)
# (요청했던 대로 원본 기준으로 출력하되, 구제된 애들도 알 수 있게 살짝 다듬었어!)
all_categories = df.select("중분류명").unique().to_series().to_list()

pb_check = df.group_by("중분류명").agg([
    pl.col("상품명").str.contains("PB").any().alias("has_pb")
])

pb_list = pb_check.filter(pl.col("has_pb")).select("중분류명").to_series().to_list()
non_pb_list = pb_check.filter(~pl.col("has_pb")).select("중분류명").to_series().to_list()

print(f"pb라는 문자열이 있는 중분류 : {pb_list} / 총 {len(pb_list)}개")
print(f"없는 중분류: {non_pb_list} / 총 {len(non_pb_list)}개")
print(f"💡 그 중 예외로 구제되어 살아남은 중분류: {exception_categories} / 총 {len(exception_categories)}개")

pb라는 문자열이 있는 중분류 : ['컵커피', '냉장안주', '냉장간편식', '샐러드', '기능성드링크', '오징어', '가공미반류', '젤리', '캔', '조리냉동', '흰우유', '냉동간편식', '농산안주', '전통음료', '얼음', '노벨티', '용기면', '스포츠음료', '프로틴 음료', '스낵류', '비스킷류', '육포', '건해산물', '프로틴/시리얼', '초콜릿', '냉장베이커리', '세븐카페', '차음료', '커피음료', '너트류', '냉장주스', '껌', '상온간편식', '간식빵', '파우치', '가공우유', '냉동만두', '젤리류', '원컵', '차', '주스', '구움과자', '스낵안주', '컨셉카페', '냉장간식', '캔디류', '요구르트'] / 총 47개
없는 중분류: ['조미료', '과일', '리쿼', '푸드 간편식', '즉석간식', '기능성우유', '소주', '국산맥주', '건강/이너뷰티', '시즌케익', '장류', '샐러드모닝랩', '도시락', '수입맥주', '즉석치킨', '식빵', '봉지면', '프리미엄', '쌀/잡곡', '분말가루', '신선냉동', '탄산음료', '생선류', '기타', '떡', '숙취해소제', '하이볼', '쨈/꿀', '고급', '냉동디저트', '기타전통주', '식용유지', '선물세트', '야채', '홈타입', '커피', '삼각김밥', '냉장피자', '소스류', '햄버거', '김밥', '양주', '유제품', '막걸리', '즉석조리', '수축산/계란', '두유', '와인', '초밥류', '샌드위치', '호빵', '무알콜맥주'] / 총 52개
💡 그 중 예외로 구제되어 살아남은 중분류: ['기타전통주', '삼각김밥', '떡', '햄버거', '냉장피자', '국산맥주', '양주', '샌드위치', '김밥', '와인', '초밥류', '도시락', '하이볼', '고급', '냉동디저트'] / 총 15개


In [4]:
import polars as pl

# 1. 데이터 경로 설정
input_path = "/Users/hajiyoon/workspace/data/processed/pareto_processed_results.csv"
output_path = "/Users/hajiyoon/workspace/data/processed/pareto_processed_results_v2.csv"

# 2. 데이터 읽기
df = pl.read_csv(input_path)

# 3. '상품명' 컬럼 왼쪽에 '세븐일레븐 ' 추가하기
# pl.lit()은 리터럴(고정된 문자열)을 의미해! 
# 고정 문자열과 기존 컬럼을 더하기(+)만 하면 바로 합쳐져.
df = df.with_columns(
    (pl.lit("세븐일레븐 ") + pl.col("상품명")).alias("상품명")
)

# 4. 수정된 데이터 저장
df.write_csv(output_path)

print(f"✅ 처리가 완료되었어! 결과는 '{output_path}'에 저장됐어.")

✅ 처리가 완료되었어! 결과는 '/Users/hajiyoon/workspace/data/processed/pareto_processed_results_v2.csv'에 저장됐어.


In [10]:
import os
import time
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import quote

# 1. 환경 변수 및 API 설정
load_dotenv()
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

# 2. 고정 설정 값
INPUT_PATH = '/Users/hajiyoon/workspace/data/processed/pareto_processed_results.csv'
OUTPUT_PATH = 'seven_eleven_full_reviews.csv'
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_blog_content(url):
    """
    네이버 블로그의 복잡한 구조를 분석하여 본문 텍스트를 추출합니다.
    """
    try:
        # 1차 접속: iframe의 src 주소 추출
        response = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        iframe = soup.find('iframe', id='mainFrame')
        if not iframe:
            return "본문 추출 실패 (iframe 없음)"
        
        # ✅ 수정 1: iframe src 절대경로 여부 분기 처리
        src = iframe['src']
        if src.startswith('http'):
            inner_url = src  # 이미 절대경로이면 그대로 사용
        else:
            inner_url = "https://blog.naver.com" + src  # 상대경로이면 도메인 붙이기
        
        # 2차 접속 전 짧은 대기 (안정성 확보)
        time.sleep(random.uniform(0.5, 1))
        
        # 2차 접속: 실제 본문 페이지 파싱
        inner_response = requests.get(inner_url, headers=HEADERS, timeout=10)
        inner_soup = BeautifulSoup(inner_response.text, 'html.parser')
        
        # 본문 태그 순차 탐색 (신버전 -> 구버전 -> 아주 오래된 버전 -> 모바일)
        # 1순위: div.se-main-container (신버전 스마트에디터 ONE)
        content_area = inner_soup.select_one('div.se-main-container')
        
        # 2순위: div#postViewArea (구버전 스마트에디터 2.0)
        if not content_area:
            content_area = inner_soup.select_one('div#postViewArea')
            
        # 3순위: div[id^='post-view'] (아주 오래된 구버전)
        if not content_area:
            content_area = inner_soup.select_one("div[id^='post-view']")
            
        # 4순위: div.se-viewer (모바일/기타)
        if not content_area:
            content_area = inner_soup.select_one('div.se-viewer')
            
        # 텍스트 추출 및 정제
        if content_area:
            text = content_area.get_text(separator='\n', strip=True)
            return text if text else "본문 추출 실패 (텍스트 없음)"
        else:
            return "본문 추출 실패 (JS 렌더링 필요)"
            
    except Exception as e:
        return f"크롤링 오류: {str(e)}"

def main():
    # .env 파일 체크
    if not CLIENT_ID or not CLIENT_SECRET:
        print("경고: .env 파일에 NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 설정되지 않았습니다.")
        if not os.path.exists('.env'):
            with open('.env', 'w') as f:
                f.write("NAVER_CLIENT_ID=\nNAVER_CLIENT_SECRET=\n")
            print(".env 파일 템플릿을 생성했습니다. API 키를 입력 후 다시 실행해 주세요.")
        return

    # 데이터 로드
    if not os.path.exists(INPUT_PATH):
        print(f"오류: '{INPUT_PATH}' 경로에 파일이 존재하지 않습니다.")
        return

    df = pd.read_csv(INPUT_PATH)
    
    # ✅ 수정 2: 변수명 items → product_list (API 응답의 'items'와 충돌 방지)
    product_list = df['상품명'].dropna().unique()
    
    results = []
    print(f"총 {len(product_list)}개의 검색 키워드에 대해 크롤링을 수행합니다.")

    for i, item in enumerate(product_list, 1):
        search_query = f"세븐일레븐 {item}"
        print(f"[{i}/{len(product_list)}] 검색어: {search_query}")
        
        # 네이버 블로그 검색 API 호출 (상위 3개, 관련도순)
        api_url = f"https://openapi.naver.com/v1/search/blog.json?query={quote(search_query)}&display=3&sort=sim"
        api_headers = {
            "X-Naver-Client-Id": CLIENT_ID,
            "X-Naver-Client-Secret": CLIENT_SECRET
        }
        
        try:
            api_res = requests.get(api_url, headers=api_headers)
            if api_res.status_code != 200:
                print(f"  - API 호출 실패 (상태 코드: {api_res.status_code})")
                continue
            
            items_list = api_res.json().get('items', [])
            
            for blog in items_list:
                # 제목 HTML 태그 제거
                title = BeautifulSoup(blog['title'], 'html.parser').get_text()
                link = blog['link']
                
                print(f"  - 크롤링 중: {title[:25]}...")
                content = get_blog_content(link)
                
                results.append({
                    '검색어': search_query,
                    '블로그제목': title,
                    '블로그링크': link,
                    '본문내용': content
                })
                
                # 블로그 크롤링 간 랜덤 대기 (1~3초)
                time.sleep(random.uniform(1, 3))
                
        except Exception as e:
            print(f"  - 검색 처리 중 오류: {e}")
            continue

    # 결과물 저장
    if results:
        result_df = pd.DataFrame(results)
        result_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
        print(f"\n작업 완료! 결과가 {OUTPUT_PATH}에 저장되었습니다. (총 {len(results)}건)")
    else:
        print("\n수집된 결과가 없습니다.")

if __name__ == "__main__":
    main()

총 1084개의 검색 키워드에 대해 크롤링을 수행합니다.
[1/1084] 검색어: 세븐일레븐 CJ)둥근햇반210g
  - 크롤링 중: [무료배송]CJ제일제당 둥근 햇반 210g 3...
  - 크롤링 중: 세븐일레븐 cj 햇반 클린 식단...
  - 크롤링 중: 홈플러스-CJ 둥근 햇반 210G*10입...
[2/1084] 검색어: 세븐일레븐 CJ)큰햇반300g
  - 크롤링 중: CJ 햇반 큰공기 300g 18개 최신세일중특...
  - 크롤링 중: CJ 햇반 큰공기, 18개, 300g 특별베스...
  - 크롤링 중: CJ 햇반 백미 큰공기 300gX30개, 30...
[3/1084] 검색어: 세븐일레븐 오뚜기)오뚜기밥210g
  - 크롤링 중: 오뚜기 든든쌀밥 210g 36개...
  - 크롤링 중: 오뚜기밥 210g 24개 칼로리 장단점 최저가...
  - 크롤링 중: 즉석밥 비교 햇반 오뚜기밥 더미식밥 210g ...
[4/1084] 검색어: 세븐일레븐 CJ)작은햇반130g
  - 크롤링 중: 세븐일레븐 cj 햇반 클린 식단...
  - 크롤링 중: 햇반 백미 작은공기 130g 추천, CJ 즉석...
  - 크롤링 중: 제일제당 CJ 햇반 작은공기 130g X 24...
[5/1084] 검색어: 세븐일레븐 오뚜기)맛있는큰밥300g
  - 크롤링 중: 오뚜기 맛있는 오뚜기밥 큰밥 300g 18개 ...
  - 크롤링 중: 오뚜기 맛있는 오뚜기밥 큰밥 300g 4개 맞...
  - 크롤링 중: 지은 밥맛 그대로! 오뚜기 큰밥 300g 24...
[6/1084] 검색어: 세븐일레븐 CJ)햇반미역국밥(컵반)
  - 크롤링 중: [세븐일레븐 편의점] cj 햇반 컵반 미역국밥...
  - 크롤링 중: CJ 햇반 컵반 미역국밥 후기｜전자레인지로 먹...
  - 크롤링 중: 식품) cj 햇반 컵반 미역국밥 훅이...
[7/1084] 검색어: 세븐일레븐 동원)양반전복죽285g
  - 크롤링 중: 동원F&B 양반 전복죽 285g...
  - 크롤링 중: 동원 )양반죽 

KeyboardInterrupt: 

In [13]:
import os
import time
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import quote

# Selenium 관련 임포트
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. 환경 변수 및 API 설정
load_dotenv()
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

# 2. 고정 설정 값
INPUT_PATH = '/Users/hajiyoon/workspace/data/processed/pareto_processed_results.csv'
OUTPUT_PATH = 'seven_eleven_full_reviews_selenium.csv'


def get_blog_content(driver, url):
    """
    Selenium을 사용하여 네이버 블로그 본문을 추출합니다.
    """
    try:
        # ✅ 수정 2: 함수 시작 시 항상 default_content로 초기화 (frame 꼬임 방지)
        driver.switch_to.default_content()

        # 페이지 접속
        driver.get(url)

        # iframe(mainFrame) 전환 대기 및 이동
        try:
            wait = WebDriverWait(driver, 10)
            wait.until(EC.presence_of_element_located((By.ID, "mainFrame")))
            driver.switch_to.frame("mainFrame")
        except:
            return "본문 추출 실패 (iframe 미확인)"

        # ✅ 수정 5: 페이지 body 로딩을 한 번만 대기한 뒤, find_elements로 즉시 탐색
        #           (셀렉터마다 3초씩 대기하던 방식 → 최대 12초 소요 문제 해결)
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
        except:
            return "본문 추출 실패 (페이지 로딩 타임아웃)"

        # 본문 태그 후보군 (순서대로 즉시 탐색)
        selectors = [
            "div.se-main-container",  # 신버전 스마트에디터 ONE
            "div#postViewArea",       # 구버전 스마트에디터 2.0
            "div[id^='post-view']",   # 아주 오래된 구버전
            "div.se-viewer"           # 모바일/기타
        ]

        for selector in selectors:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if elements:
                content_text = elements[0].text.strip()
                if content_text:
                    return content_text

        return "본문 추출 실패 (구조 미확인)"

    except Exception as e:
        return f"크롤링 오류: {str(e)}"
    finally:
        # iframe에서 기본 컨텐츠로 복귀 (다음 크롤링을 위해 중요)
        try:
            driver.switch_to.default_content()
        except:
            pass


def setup_driver():
    """
    Mac M1/M2 환경에 최적화된 Selenium Chrome 드라이버를 설정합니다.
    """
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver


def main():
    # .env 파일 체크
    if not CLIENT_ID or not CLIENT_SECRET:
        print("오류: .env 파일에 NAVER_CLIENT_ID 및 NAVER_CLIENT_SECRET이 필요합니다.")
        return

    # 데이터 로드
    if not os.path.exists(INPUT_PATH):
        print(f"오류: '{INPUT_PATH}' 파일을 찾을 수 없습니다.")
        return

    df = pd.read_csv(INPUT_PATH)

    # ✅ 수정 1: 변수명 items → product_list (API 응답의 'items'와 혼동·충돌 방지)
    product_list = df['상품명'].dropna().unique()

    results = []

    # ✅ 수정 6: API 호출 횟수 카운팅 초기화 (하루 25,000건 한도 추적)
    api_call_count = 0

    print("크롬 드라이버를 시작합니다 (Headless 모드)...")
    driver = setup_driver()

    try:
        print(f"총 {len(product_list)}개의 상품 키워드 크롤링 시작...")

        for i, item in enumerate(product_list, 1):
            search_query = f"세븐일레븐 {item}"
            print(f"[{i}/{len(product_list)}] 검색어: {search_query}")

            # 네이버 블로그 검색 API 호출
            api_url = (
                f"https://openapi.naver.com/v1/search/blog.json"
                f"?query={quote(search_query)}&display=3&sort=sim"
            )
            api_headers = {
                "X-Naver-Client-Id": CLIENT_ID,
                "X-Naver-Client-Secret": CLIENT_SECRET
            }

            try:
                res = requests.get(api_url, headers=api_headers)

                # ✅ 수정 6: API 호출마다 카운트 증가 및 출력
                api_call_count += 1
                print(f"  - 누적 API 호출 횟수: {api_call_count}건 / 일일 한도 25,000건")

                if res.status_code != 200:
                    print(f"  - API 호출 실패 (Code: {res.status_code})")
                    continue

                blog_items = res.json().get('items', [])

                for blog in blog_items:
                    title = BeautifulSoup(blog['title'], 'html.parser').get_text()
                    link = blog['link']

                    # ✅ 수정 3: 네이버 블로그 외 링크 사전 필터링 (불필요한 Selenium 호출 방지)
                    if 'blog.naver.com' not in link:
                        print(f"  - 네이버 블로그 외 링크 제외: {link}")
                        results.append({
                            '검색어': search_query,
                            '블로그제목': title,
                            '블로그링크': link,
                            '본문내용': '네이버 블로그 외 링크 제외'
                        })
                        continue

                    print(f"  - 셀레니움 크롤링 중: {title[:20]}...")
                    content = get_blog_content(driver, link)

                    results.append({
                        '검색어': search_query,
                        '블로그제목': title,
                        '블로그링크': link,
                        '본문내용': content
                    })

                    time.sleep(random.uniform(1, 3))

            except Exception as e:
                print(f"  - 처리 중 예외 발생: {e}")
                continue

            # ✅ 수정 4: 10개 상품마다 중간 저장 (오류 발생 시 데이터 유실 방지)
            if i % 10 == 0 and results:
                pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
                print(f"  - 중간 저장 완료 ({i}번째 / 누적 {len(results)}건)")

    finally:
        print("\n드라이버를 종료합니다.")
        driver.quit()

    # 최종 결과물 저장
    if results:
        result_df = pd.DataFrame(results)
        result_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
        print(f"\n작업 완료! 결과가 {OUTPUT_PATH}에 저장되었습니다. (총 {len(results)}건)")
        print(f"총 API 호출 횟수: {api_call_count}건")
    else:
        print("\n수집된 데이터가 없습니다.")


if __name__ == "__main__":
    main()

크롬 드라이버를 시작합니다 (Headless 모드)...
총 1084개의 상품 키워드 크롤링 시작...
[1/1084] 검색어: 세븐일레븐 CJ)둥근햇반210g
  - 누적 API 호출 횟수: 1건 / 일일 한도 25,000건
  - 셀레니움 크롤링 중: [무료배송]CJ제일제당 둥근 햇반 2...
  - 셀레니움 크롤링 중: 세븐일레븐 cj 햇반 클린 식단...
  - 셀레니움 크롤링 중: 홈플러스-CJ 둥근 햇반 210G*1...
[2/1084] 검색어: 세븐일레븐 CJ)큰햇반300g
  - 누적 API 호출 횟수: 2건 / 일일 한도 25,000건
  - 셀레니움 크롤링 중: CJ 햇반 큰공기 300g 18개 최...
  - 셀레니움 크롤링 중: CJ 햇반 큰공기, 18개, 300g...
  - 셀레니움 크롤링 중: CJ 햇반 백미 큰공기 300gX30...
[3/1084] 검색어: 세븐일레븐 오뚜기)오뚜기밥210g
  - 누적 API 호출 횟수: 3건 / 일일 한도 25,000건
  - 셀레니움 크롤링 중: 오뚜기 든든쌀밥 210g 36개...
  - 셀레니움 크롤링 중: 오뚜기밥 210g 24개 칼로리 장단...
  - 셀레니움 크롤링 중: 즉석밥 비교 햇반 오뚜기밥 더미식밥 ...
[4/1084] 검색어: 세븐일레븐 CJ)작은햇반130g
  - 누적 API 호출 횟수: 4건 / 일일 한도 25,000건
  - 셀레니움 크롤링 중: 세븐일레븐 cj 햇반 클린 식단...
  - 셀레니움 크롤링 중: 햇반 백미 작은공기 130g 추천, ...
  - 셀레니움 크롤링 중: 제일제당 CJ 햇반 작은공기 130g...
[5/1084] 검색어: 세븐일레븐 오뚜기)맛있는큰밥300g
  - 누적 API 호출 횟수: 5건 / 일일 한도 25,000건
  - 셀레니움 크롤링 중: 오뚜기 맛있는 오뚜기밥 큰밥 300g...
  - 셀레니움 크롤링 중: 오뚜기 맛있는 오뚜기밥 큰밥 300g...
  - 셀레니움 크롤링 중: 지은 밥맛 그대로! 오뚜기 큰밥 30...
[6/1

In [2]:
import os
import time
import random
import pandas as pd
import polars as pl
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. 환경 변수 및 설정
load_dotenv()
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")
INPUT_DATA_PATH = "/Users/hajiyoon/workspace/data/processed/최종_상품단위_데이터셋.csv"
OUTPUT_PATH = "seven_eleven_remaining_80_reviews.csv"

def get_remaining_products():
    """
    중분류별 매출 누적 비율 80% 이상(매출 기여도가 낮은 나머지 상품군)인 상품들 추출
    """
    print("데이터 로딩 및 하위 매출 상품군(나머지 80% 상품수 기준) 추출 중...")
    
    # 데이터가 큰 경우를 대비해 polars 사용
    lf = pl.scan_csv(INPUT_DATA_PATH)
    
    df = lf.select([
        pl.col("상품중분류명").alias("중분류명"),
        pl.col("상품명"),
        pl.col("총_매출금액").alias("총매출금액")
    ]).collect()

    # 중분류별 총 매출액 계산
    df = df.with_columns([
        pl.col("총매출금액").sum().over("중분류명").alias("중분류총매출금액")
    ])
    
    # 매출 내림차순 정렬 후 누적 비율 계산
    df = df.sort(["중분류명", "총매출금액"], descending=[False, True])
    df = df.with_columns([
        (pl.col("총매출금액").cum_sum().over("중분류명") / pl.col("중분류총매출금액")).alias("누적비율")
    ])

    # 기존 작업(상위 80% 매출 견인 상품)의 반대 조건
    # 누적 비율 0.8 이상인 상품들이 나머지 매출 20%를 구성하는 다수의 상품들임
    df_remaining = df.filter(pl.col("누적비율") >= 0.8)
    
    # 고유 상품명 리스트 반환
    return df_remaining.select("상품명").unique().to_series().to_list()

def setup_driver():
    """
    Selenium Chrome 드라이버 설정
    """
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

def get_blog_content(driver, url):
    """
    Selenium을 사용하여 블로그 본문 추출
    """
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 5)
        wait.until(EC.presence_of_element_located((By.ID, "mainFrame")))
        driver.switch_to.frame("mainFrame")
        
        selectors = [
            "div.se-main-container",
            "div#postViewArea",
            "div[id^='post-view']",
            "div.se-viewer"
        ]
        
        for selector in selectors:
            try:
                element = driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text
            except:
                continue
        return "본문 추출 실패"
    except Exception as e:
        return f"크롤링 오류: {e}"
    finally:
        try:
            driver.switch_to.default_content()
        except:
            pass

def main():
    if not CLIENT_ID or not CLIENT_SECRET:
        print("오류: .env 파일에 NAVER_CLIENT_ID 및 NAVER_CLIENT_SECRET 설정이 필요합니다.")
        return

    # 1. 대상 상품 추출
    items = get_remaining_products()
    print(f"추출된 대상 상품 수: {len(items)}개")
    
    results = []
    driver = setup_driver()
    
    try:
        for i, item in enumerate(items, 1):
            search_query = f"세븐일레븐 {item}"
            print(f"[{i}/{len(items)}] 검색 중: {search_query}")
            
            # 2. 네이버 블로그 검색 API (상위 3개)
            api_url = f"https://openapi.naver.com/v1/search/blog.json?query={quote(search_query)}&display=3&sort=sim"
            api_headers = {
                "X-Naver-Client-Id": CLIENT_ID,
                "X-Naver-Client-Secret": CLIENT_SECRET
            }
            
            try:
                res = requests.get(api_url, headers=api_headers)
                if res.status_code == 200:
                    blog_items = res.json().get('items', [])
                    for blog in blog_items:
                        title = BeautifulSoup(blog['title'], 'html.parser').get_text()
                        link = blog['link']
                        
                        print(f"  - 본문 수집 중: {title[:20]}...")
                        content = get_blog_content(driver, link)
                        
                        results.append({
                            '검색어': search_query,
                            '상품명': item,
                            '블로그제목': title,
                            '블로그링크': link,
                            '본문내용': content
                        })
                        time.sleep(random.uniform(1, 2))
                else:
                    print(f"  - API 호출 실패 (상태코드: {res.status_code})")
            except Exception as e:
                print(f"  - 검색 처리 중 오류: {e}")
            
            # 3. 주기적 저장 (10개 상품마다)
            if i % 10 == 0 and results:
                pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
                print(f"--- 현재까지 {len(results)}건 저장 완료 ---")
                
    finally:
        driver.quit()
        if results:
            pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
            print(f"\n최종 완료! 총 {len(results)}건의 데이터가 {OUTPUT_PATH}에 저장되었습니다.")
        else:
            print("\n수집된 데이터가 없습니다.")

if __name__ == "__main__":
    main()


데이터 로딩 및 하위 매출 상품군(나머지 80% 상품수 기준) 추출 중...
추출된 대상 상품 수: 8196개
[1/8196] 검색 중: 세븐일레븐 7P)스트롱사와자몽500ml
  - 본문 수집 중: 세븐일레븐 스트롱사와 자몽, 라임 후...
  - 본문 수집 중: 세븐일레븐 스트롱사와 자몽 레몬 시콰...
  - 본문 수집 중: [세븐일레븐신상] 일본 하이볼 스트롱...
[2/8196] 검색 중: 세븐일레븐 삼립)라이츄슈크림데니쉬100g
  - 본문 수집 중: 포켓몬빵 신상 삼립 라이츄의 슈크림데...
  - 본문 수집 중: 삼립. 라이츄 슈크림 데니쉬 후기 -...
  - 본문 수집 중: [일상] 삼립 포켓몬빵 라이츄의 슈크...
[3/8196] 검색 중: 세븐일레븐 APP전용)안유성명장샌드
  - 본문 수집 중: [금주의 신상] 스타벅스 X 해리포터...
[4/8196] 검색 중: 세븐일레븐 던_오리온)참붕어빵8P
  - 본문 수집 중: 경제 일반..네이버(언론사제공) 20...
[5/8196] 검색 중: 세븐일레븐 일신)클래식쌀쿠키300g
  - 본문 수집 중: 9월 편의점 신상 모음 CU GS25...
  - 본문 수집 중: 일신)클래식 쌀쿠키 & 위스트) 크크...
  - 본문 수집 중: [세븐일레븐 신상과자] 클래식 쌀과자...
[6/8196] 검색 중: 세븐일레븐 롯데)와 바닐라190ml
  - 본문 수집 중: 세븐일레븐 7월행사 추천 7ELEVE...
  - 본문 수집 중: 7ELEVEN 세븐일레븐 6월행사 이...
  - 본문 수집 중: 세븐일레븐 8월행사 추천 1+1 2+...
[7/8196] 검색 중: 세븐일레븐 농심)카구리큰사발
  - 본문 수집 중: [농심] 카구리 큰사발...
  - 본문 수집 중: [CU신상 농심 푸팟퐁구리 큰사발] ...
  - 본문 수집 중: [농심] 카구리 큰사발 컵라면...
[8/8196] 검색 중: 세븐일레븐 PB)복숭아아이스티1.5L
  - 본문 수집 중: 가평계곡, 유명산 옛집 오리로스, 세...
  -

KeyboardInterrupt: 

In [4]:
import os
import time
import random
import pandas as pd
import polars as pl
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm  # 진행률 바 추가!

# 1. 환경 변수 및 설정
load_dotenv()
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")
INPUT_DATA_PATH = "/Users/hajiyoon/workspace/data/processed/최종_상품단위_데이터셋.csv"
OUTPUT_PATH = "seven_eleven_remaining_80_reviews.csv"

def get_remaining_products():
    """
    중분류별 매출 누적 비율 80% 이상(매출 기여도가 낮은 나머지 상품군)인 상품들 추출
    """
    print("데이터 로딩 및 하위 매출 상품군(나머지 80% 상품수 기준) 추출 중...")
    
    lf = pl.scan_csv(INPUT_DATA_PATH)
    
    df = lf.select([
        pl.col("상품중분류명").alias("중분류명"),
        pl.col("상품명"),
        pl.col("총_매출금액").alias("총매출금액")
    ]).collect()

    df = df.with_columns([
        pl.col("총매출금액").sum().over("중분류명").alias("중분류총매출금액")
    ])
    
    df = df.sort(["중분류명", "총매출금액"], descending=[False, True])
    df = df.with_columns([
        (pl.col("총매출금액").cum_sum().over("중분류명") / pl.col("중분류총매출금액")).alias("누적비율")
    ])

    df_remaining = df.filter(pl.col("누적비율") >= 0.8)
    return df_remaining.select("상품명").unique().to_series().to_list()

def setup_driver():
    """
    Selenium Chrome 드라이버 설정
    """
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

def get_blog_content(driver, url):
    """
    Selenium을 사용하여 블로그 본문 추출
    """
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 5)
        wait.until(EC.presence_of_element_located((By.ID, "mainFrame")))
        driver.switch_to.frame("mainFrame")
        
        selectors = [
            "div.se-main-container",
            "div#postViewArea",
            "div[id^='post-view']",
            "div.se-viewer"
        ]
        
        for selector in selectors:
            try:
                element = driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text
            except:
                continue
        return "본문 추출 실패"
    except Exception as e:
        return f"크롤링 오류: {e}"
    finally:
        try:
            driver.switch_to.default_content()
        except:
            pass

def main():
    if not CLIENT_ID or not CLIENT_SECRET:
        print("오류: .env 파일에 NAVER_CLIENT_ID 및 NAVER_CLIENT_SECRET 설정이 필요합니다.")
        return

    # 1. 대상 상품 추출
    items = get_remaining_products()
    
    # 🌟 이어달리기 로직: 특정 상품 찾아서 그 다음부터 자르기
    resume_item = "롯데)제로쿠키&크림바80ml"
    
    if resume_item in items:
        start_idx = items.index(resume_item) + 1 # 완료된 상품 바로 다음 인덱스
        items = items[start_idx:]
        print(f"\n✅ '{resume_item}' 이후부터 이어서 작업을 시작합니다.")
    else:
        print(f"\n⚠️ '{resume_item}'을 목록에서 찾을 수 없어 처음부터 진행합니다.")
        
    print(f"앞으로 수집할 남은 상품 수: {len(items)}개\n")
    
    batch_results = [] # 10개씩 모아서 저장하고 비울 임시 바구니
    driver = setup_driver()
    
    try:
        # tqdm으로 묶어서 예쁜 진행률 바 생성
        for i, item in enumerate(tqdm(items, desc="크롤링 진행률", ncols=100), 1):
            search_query = f"세븐일레븐 {item}"
            
            api_url = f"https://openapi.naver.com/v1/search/blog.json?query={quote(search_query)}&display=3&sort=sim"
            api_headers = {
                "X-Naver-Client-Id": CLIENT_ID,
                "X-Naver-Client-Secret": CLIENT_SECRET
            }
            
            try:
                res = requests.get(api_url, headers=api_headers)
                if res.status_code == 200:
                    blog_items = res.json().get('items', [])
                    for blog in blog_items:
                        title = BeautifulSoup(blog['title'], 'html.parser').get_text()
                        link = blog['link']
                        
                        content = get_blog_content(driver, link)
                        
                        batch_results.append({
                            '검색어': search_query,
                            '상품명': item,
                            '블로그제목': title,
                            '블로그링크': link,
                            '본문내용': content
                        })
                        time.sleep(random.uniform(1, 2))
                else:
                    tqdm.write(f"⚠️ API 호출 실패 (상품명: {item}, 상태코드: {res.status_code})")
            except Exception as e:
                tqdm.write(f"⚠️ 검색 처리 중 오류 (상품명: {item}): {e}")
            
            # 3. 주기적 이어쓰기 (10개 상품마다)
            if i % 10 == 0 and batch_results:
                df_batch = pd.DataFrame(batch_results)
                
                # 기존 파일이 없으면 새로 만들고, 있으면 아래에 추가(append)
                if not os.path.exists(OUTPUT_PATH):
                    df_batch.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
                else:
                    df_batch.to_csv(OUTPUT_PATH, mode='a', index=False, header=False, encoding='utf-8-sig')
                
                batch_results.clear() # ⭐️ 덮어쓰기 방지: 다 썼으면 바구니 비우기
                
    finally:
        driver.quit()
        # 마지막으로 남은 데이터 털어넣기
        if batch_results:
            df_batch = pd.DataFrame(batch_results)
            if not os.path.exists(OUTPUT_PATH):
                df_batch.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
            else:
                df_batch.to_csv(OUTPUT_PATH, mode='a', index=False, header=False, encoding='utf-8-sig')
                
        print(f"\n🎉 크롤링 종료! 데이터가 {OUTPUT_PATH}에 안전하게 이어쓰기 되었습니다.")

if __name__ == "__main__":
    main()


데이터 로딩 및 하위 매출 상품군(나머지 80% 상품수 기준) 추출 중...

✅ '롯데)제로쿠키&크림바80ml' 이후부터 이어서 작업을 시작합니다.
앞으로 수집할 남은 상품 수: 1062개



크롤링 진행률: 100%|██████████████████████████████████████████| 1062/1062 [2:19:36<00:00,  7.89s/it]



🎉 크롤링 종료! 데이터가 seven_eleven_remaining_80_reviews.csv에 안전하게 이어쓰기 되었습니다.


In [5]:
import os
import time
import random
import pandas as pd
import polars as pl
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm

# 1. 환경 변수 및 설정
load_dotenv()
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")
INPUT_DATA_PATH = "/Users/hajiyoon/workspace/data/processed/최종_상품단위_데이터셋.csv"
OUTPUT_PATH = "seven_eleven_remaining_80_reviews.csv"

def get_remaining_products():
    print("데이터 로딩 및 하위 매출 상품군 추출 중...")
    lf = pl.scan_csv(INPUT_DATA_PATH)
    df = lf.select([
        pl.col("상품중분류명").alias("중분류명"),
        pl.col("상품명"),
        pl.col("총_매출금액").alias("총매출금액")
    ]).collect()

    df = df.with_columns([
        pl.col("총매출금액").sum().over("중분류명").alias("중분류총매출금액")
    ])
    
    df = df.sort(["중분류명", "총매출금액"], descending=[False, True])
    df = df.with_columns([
        (pl.col("총매출금액").cum_sum().over("중분류명") / pl.col("중분류총매출금액")).alias("누적비율")
    ])

    df_remaining = df.filter(pl.col("누적비율") >= 0.8)
    return df_remaining.select("상품명").unique().to_series().to_list()

def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

def get_blog_content(driver, url):
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 5)
        wait.until(EC.presence_of_element_located((By.ID, "mainFrame")))
        driver.switch_to.frame("mainFrame")
        
        selectors = [
            "div.se-main-container",
            "div#postViewArea",
            "div[id^='post-view']",
            "div.se-viewer"
        ]
        
        for selector in selectors:
            try:
                element = driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text
            except:
                continue
        return "본문 추출 실패"
    except Exception as e:
        return f"크롤링 오류: {e}"
    finally:
        try:
            driver.switch_to.default_content()
        except:
            pass

def main():
    if not CLIENT_ID or not CLIENT_SECRET:
        print("오류: .env 파일에 API 키 설정이 필요합니다.")
        return

    items = get_remaining_products()
    
    start_item = "효성)한주본소금1kg"
    end_item = "롯데)제로쿠키&크림바80ml"
    
    if start_item in items and end_item in items:
        start_idx = items.index(start_item)
        end_idx = items.index(end_item)
        
        if start_idx > end_idx:
            start_idx, end_idx = end_idx, start_idx
            
        items = items[start_idx : end_idx + 1]
        print(f"\n✅ '{start_item}'부터 '{end_item}'까지 총 {len(items)}개의 상품을 다시 수집합니다.")
    else:
        print(f"\n⚠️ 목록에서 상품을 찾을 수 없습니다. 상품명을 다시 확인해 주세요.")
        return

    # 🌟 새롭게 추가된 핵심 로직: 기존 파일에서 '잘못된 데이터' 미리 삭제하기!
    if os.path.exists(OUTPUT_PATH):
        print("\n🧹 크롤링 시작 전, 기존 파일에서 잘못 수집된 옛날 데이터를 청소합니다...")
        try:
            # 1. 기존 파일을 읽어옴
            existing_df = pd.read_csv(OUTPUT_PATH)
            original_len = len(existing_df)
            
            # 2. 이번에 수집할 'items' 리스트에 속하지 않은(~) 정상 데이터만 남김
            cleaned_df = existing_df[~existing_df['상품명'].isin(items)]
            
            # 3. 청소된 데이터로 덮어쓰기 저장
            cleaned_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
            
            deleted_count = original_len - len(cleaned_df)
            print(f"✨ 청소 완료! 잘못된 데이터 {deleted_count}줄을 삭제했습니다.")
            print(f"📊 현재 남은 정상 데이터 수: {len(cleaned_df)}줄\n")
            
        except Exception as e:
            print(f"⚠️ 기존 파일 청소 중 오류 발생: {e}")
            return # 청소하다 에러나면 혹시 모르니 멈춤

    batch_results = []
    driver = setup_driver()
    
    try:
        for i, item in enumerate(tqdm(items, desc="구간 복구 진행률", ncols=100), 1):
            search_query = f"세븐일레븐 {item}"
            
            api_url = f"https://openapi.naver.com/v1/search/blog.json?query={quote(search_query)}&display=3&sort=sim"
            api_headers = {
                "X-Naver-Client-Id": CLIENT_ID,
                "X-Naver-Client-Secret": CLIENT_SECRET
            }
            
            try:
                res = requests.get(api_url, headers=api_headers)
                if res.status_code == 200:
                    blog_items = res.json().get('items', [])
                    for blog in blog_items:
                        title = BeautifulSoup(blog['title'], 'html.parser').get_text()
                        link = blog['link']
                        
                        content = get_blog_content(driver, link)
                        
                        batch_results.append({
                            '검색어': search_query,
                            '상품명': item,
                            '블로그제목': title,
                            '블로그링크': link,
                            '본문내용': content
                        })
                        time.sleep(random.uniform(1, 2))
                else:
                    tqdm.write(f"⚠️ API 실패 (상품: {item})")
            except Exception as e:
                tqdm.write(f"⚠️ 검색 오류 (상품: {item}): {e}")
            
            if i % 10 == 0 and batch_results:
                df_batch = pd.DataFrame(batch_results)
                # 청소된 파일 밑에 안전하게 이어쓰기
                df_batch.to_csv(OUTPUT_PATH, mode='a', index=False, header=False, encoding='utf-8-sig')
                batch_results.clear()
                
    finally:
        driver.quit()
        if batch_results:
            df_batch = pd.DataFrame(batch_results)
            df_batch.to_csv(OUTPUT_PATH, mode='a', index=False, header=False, encoding='utf-8-sig')
                
        print(f"\n🎉 구간 복구 완료! 깨끗한 새 데이터가 {OUTPUT_PATH}에 완벽하게 교체/저장되었습니다.")

if __name__ == "__main__":
    main()


데이터 로딩 및 하위 매출 상품군 추출 중...

✅ '효성)한주본소금1kg'부터 '롯데)제로쿠키&크림바80ml'까지 총 3735개의 상품을 다시 수집합니다.

🧹 크롤링 시작 전, 기존 파일에서 잘못 수집된 옛날 데이터를 청소합니다...
✨ 청소 완료! 잘못된 데이터 6710줄을 삭제했습니다.
📊 현재 남은 정상 데이터 수: 8143줄



구간 복구 진행률: 100%|███████████████████████████████████████| 3735/3735 [8:02:18<00:00,  7.75s/it]



🎉 구간 복구 완료! 깨끗한 새 데이터가 seven_eleven_remaining_80_reviews.csv에 완벽하게 교체/저장되었습니다.


In [6]:
import pandas as pd
import os

# 파일 경로 설정
ORIGINAL_FILE = "seven_eleven_remaining_80_reviews.csv"
ERROR_FILE = "seven_eleven_error_products.csv" # 불량 데이터만 모아둘 새 파일!

def separate_clean_and_error_data():
    if not os.path.exists(ORIGINAL_FILE):
        print(f"⚠️ {ORIGINAL_FILE} 파일이 같은 폴더에 없는 것 같아요!")
        return

    print("🔍 기존 데이터를 스캔하여 '크롤링 오류'를 솎아냅니다...\n")
    df = pd.read_csv(ORIGINAL_FILE)
    
    # 🌟 '크롤링 오류'나 '본문 추출 실패'로 시작하는 불량 데이터 색출!
    error_mask = df['본문내용'].astype(str).str.startswith('크롤링 오류') | df['본문내용'].astype(str).str.startswith('본문 추출 실패')
    
    error_df = df[error_mask]
    clean_df = df[~error_mask]
    
    if error_df.empty:
        print("🎉 대박! 오류 난 데이터가 하나도 없습니다. 바로 팀원들에게 보내도 돼요!")
        return
        
    unique_error_items = error_df['상품명'].unique()
    
    # 1. 깨끗한 데이터는 기존 파일 이름으로 덮어쓰기 (팀원 전달용 🎁)
    clean_df.to_csv(ORIGINAL_FILE, index=False, encoding='utf-8-sig')
    print(f"✅ [청소 완료] 정상 데이터 {len(clean_df)}줄만 남겨서 '{ORIGINAL_FILE}'에 덮어썼습니다.")
    print("   👉 이 파일을 바로 팀원들에게 전달하시면 됩니다!")

    # 2. 에러난 데이터는 새로운 파일로 빼두기 (나중에 복구할 용도 🛠️)
    error_df.to_csv(ERROR_FILE, index=False, encoding='utf-8-sig')
    print(f"\n🚨 [격리 완료] 오류 데이터 {len(error_df)}줄은 '{ERROR_FILE}'에 따로 저장했습니다.")
    print(f"   👉 나중에 다시 크롤링해야 할 상품 종류는 총 {len(unique_error_items)}개입니다.")

if __name__ == "__main__":
    separate_clean_and_error_data()


🔍 기존 데이터를 스캔하여 '크롤링 오류'를 솎아냅니다...

✅ [청소 완료] 정상 데이터 11933줄만 남겨서 'seven_eleven_remaining_80_reviews.csv'에 덮어썼습니다.
   👉 이 파일을 바로 팀원들에게 전달하시면 됩니다!

🚨 [격리 완료] 오류 데이터 6563줄은 'seven_eleven_error_products.csv'에 따로 저장했습니다.
   👉 나중에 다시 크롤링해야 할 상품 종류는 총 2279개입니다.


## 이거 하면 돼!!!!!!!!

In [1]:
import pandas as pd 
import numpy as np

df = pd.read_csv('C:\\Users\\alexj\\2026_캡스톤_디자인\\세븐일레븐_프로젝트\\세븐일레븐_내부데이터\\전처리_EDA\\최종\\pos_data_food_final_상품단위.csv')
print(df.shape)

target_cat = ['컵커피', '냉장안주', '냉장간편식', '샐러드', '기능성드링크', '오징어', '가공미반류', '젤리', '캔', '조리냉동', '흰우유', '냉동간편식', '농산안주', '전통음료', '얼음', '노벨티', '용기면', '스포츠음료', '프로틴 음료', '스낵류', '비스킷류', '육포', '건해산물', '프로틴/시리얼', '초콜릿', '냉장베이커리', '세븐카페', '차음료', '커피음료', '너트류', '냉장주스', '껌', '상온간편식', '간식빵', '파우치', '가공우유', '냉동만두', '젤리류', '원컵', '차', '주스', '구움과자', '스낵안주', '컨셉카페', '냉장간식', '캔디류', '요구르트',
        '기타전통주', '삼각김밥', '떡', '햄버거', '냉장피자', '국산맥주', '양주', '샌드위치', '김밥', '와인', '초밥류', '도시락', '하이볼', '고급', '냉동디저트']

# strip() 후 필터링
filtered_df = df[df['상품중분류명'].str.strip().isin(target_cat)].copy()
print(f"필터링 후: {filtered_df.shape}")
# 저장
output_path = r'C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\전처리_EDA\최종\pos_data_food_final_상품단위_filtered.csv'
filtered_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print("저장 완료:", output_path)

(10579, 26)
필터링 후: (8186, 26)
저장 완료: C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\전처리_EDA\최종\pos_data_food_final_상품단위_filtered.csv


In [2]:
success = pd.read_csv('C:\\Users\\alexj\\2026_캡스톤_디자인\\세븐일레븐_프로젝트\\세븐일레븐_내부데이터\\seven_eleven_blog_reviews_success.csv')
print(success.shape)
success.head(1)

(3181, 4)


,검색어,블로그제목,블로그링크,본문내용
0,세븐일레븐 CJ)둥근햇반210g,[무료배송]CJ제일제당 둥근 햇반 210g 36개,https://blog.naver.com/ccgu4745/223751684668,[11번가] [무료배송]CJ제일제당 둥근 햇반 210g 36개\n냉장/냉동식품>냉장...


In [4]:
success.isnull().sum()

검색어      0
블로그제목    0
블로그링크    0
본문내용     0
dtype: int64

In [3]:
fail = pd.read_csv('C:\\Users\\alexj\\2026_캡스톤_디자인\\세븐일레븐_프로젝트\\세븐일레븐_내부데이터\\seven_eleven_blog_reviews_fail_v1.csv')
print(fail.shape)
fail.head(1)

(74932, 6)


,중분류명,검색어,상품명,블로그제목,블로그링크,본문내용
0,하이볼,세븐일레븐 7P)스트롱사와자몽500ml,7P)스트롱사와자몽500ml,"세븐일레븐 스트롱사와 자몽, 라임 후기 과일 맥주 추천",https://blog.naver.com/lovekra88/223406679573,안녕하세요\n부사오지상 현대리입니다\n오늘은 얼마전 마셨던\n세븐일레븐 스트롱사와\...


In [5]:
fail.isnull().sum()

중분류명         0
검색어          0
상품명          0
블로그제목     3269
블로그링크     3269
본문내용     66075
dtype: int64

In [3]:
import sys
!{sys.executable} -m pip install selenium webdriver-manager

  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -------------------- ------------------- 5.0/9.6 MB 23.2 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 24.5 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 21.3 MB/s eta 0:00:00
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Attempting uninstall: websocket-client
    Found existing installation: websocket-client 0.58.0
    Uninstalling websocket-client-0.58.0:
      Successfully uninstalled we

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
botocore 1.29.76 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
spyder 5.4.3 requires jedi<0.19.0,>=0.17.2, but you have jedi 0.19.2 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import time
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from queue import Queue

# ─────────────────────────────────────────────
# 1. 환경 변수 및 경로 설정
# ─────────────────────────────────────────────
load_dotenv()
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")
MASTER_PATH = "C:\\Users\\alexj\\2026_캡스톤_디자인\\세븐일레븐_프로젝트\\세븐일레븐_내부데이터\\전처리_EDA\\최종\\pos_data_food_final_상품단위_filtered.csv"
SUCCESS_PATH = "C:\\Users\\alexj\\2026_캡스톤_디자인\\세븐일레븐_프로젝트\\세븐일레븐_내부데이터\\seven_eleven_blog_reviews_success.csv"
FAIL_PATH = "C:\\Users\\alexj\\2026_캡스톤_디자인\\세븐일레븐_프로젝트\\세븐일레븐_내부데이터\\seven_eleven_blog_reviews_fail_v1.csv"
OUTPUT_PATH = 'C:\\Users\\alexj\\2026_캡스톤_디자인\\세븐일레븐_프로젝트\\세븐일레븐_내부데이터\\블로그_나머지상품_크롤링.csv'

# 동시 실행할 드라이버 수 (RAM 여유분 / 300MB 로 조절. 기본값 5)
DRIVER_COUNT = 5

# ───────────────────────────────────────────
# 2. 이어서 크롤링할 대상 추출
#    → 이미 FAIL / SUCCESS 파일에 기록된 검색어는 모두 제외
# ─────────────────────────────────────────────
def get_extra_target_items():
    print("수집 대상 아이템 분석 중...")
    # ── MASTER에서 전체 유니크 상품명 추출 ──────────────────
    df_master = pd.read_csv(MASTER_PATH, low_memory=False)
    master_items = set(df_master['상품명'].dropna().str.strip().unique())
    print(f"MASTER 전체 유니크 상품명: {len(master_items)}개")
    # ── 이미 크롤링 완료된 상품명 수집 (본문내용이 있는 경우만) ──
    done_items = set()
    for path, label in [(SUCCESS_PATH, 'success'), (FAIL_PATH, 'fail')]:
        if not os.path.exists(path):
            print(f"[{label}] 파일 없음, 스킵")
            continue
        df_temp = pd.read_csv(path, low_memory=False)
        if '검색어' not in df_temp.columns:
            print(f"[{label}] '검색어' 컬럼 없음, 스킵")
            continue
        # '본문내용'이 null이 아닌 것만 → 완료로 간주
        if '본문내용' in df_temp.columns:
            df_done = df_temp[df_temp['본문내용'].notna()]
        else:
            df_done = df_temp  # 컬럼 자체가 없으면 전부 완료로 간주
        items = (
            df_done['검색어']
            .dropna()
            .str.replace('세븐일레븐 ', '', n=1, regex=False)
            .str.strip()
        )
        done_items.update(items.tolist())
        print(f"[{label}] 완료 처리된 항목: {len(items)}개")
    print(f"총 완료(제외할) 항목 수: {len(done_items)}개")
    # ── MASTER 기준으로 남은 크롤링 대상 확정 ──────────────────
    df_remaining = df_master[
        df_master['상품명'].str.strip().isin(master_items - done_items)
    ][['상품중분류명', '상품명']].drop_duplicates()
    print(f"남은 크롤링 대상: {len(df_remaining)}개\n")
    return df_remaining.values.tolist()  # → [(카테고리, 상품명), ...] 튜플 리스트

# ─────────────────────────────────────────────
# 3. 드라이버 설정 (이미지/CSS/폰트 차단으로 로딩 가속)
# ─────────────────────────────────────────────
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--dns-prefetch-disable")
    chrome_options.add_argument("--blink-settings=imagesEnabled=false")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    chrome_options.add_experimental_option("prefs", {
        "profile.managed_default_content_settings.images": 2,      # 이미지 차단
        "profile.managed_default_content_settings.stylesheets": 2, # CSS 차단
        "profile.managed_default_content_settings.fonts": 2,       # 폰트 차단
    })

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=chrome_options)


def create_driver_pool(n):
    """드라이버 n개를 Queue에 담아 반환"""
    pool = Queue()
    print(f"드라이버 {n}개 초기화 중...")
    for i in range(n):
        pool.put(setup_driver())
        print(f"  드라이버 {i+1}/{n} 준비 완료")
    return pool


# ─────────────────────────────────────────────
# 4. 블로그 본문 추출 (Selenium, 타임아웃 단축)
# ─────────────────────────────────────────────
def get_blog_content(driver, url):
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 3)  # 5 → 3초
        wait.until(EC.presence_of_element_located((By.ID, "mainFrame")))
        driver.switch_to.frame("mainFrame")

        for selector in ["div.se-main-container", "div#postViewArea", "div.se-viewer"]:
            try:
                element = driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text
            except:
                continue
        return "본문 추출 실패"
    except:
        return "크롤링 오류"
    finally:
        try:
            driver.switch_to.default_content()
        except:
            pass


# ─────────────────────────────────────────────
# 5. 상품 1개 처리 (스레드 단위 작업)
# ─────────────────────────────────────────────
def process_item(driver_pool, cat, item):
    driver = driver_pool.get()  # 드라이버 대여
    results = []

    try:
        item_str = str(item)
        search_query = item_str if "세븐일레븐" in item_str else f"세븐일레븐 {item_str}"

        api_url = (
            f"https://openapi.naver.com/v1/search/blog.json"
            f"?query={quote(search_query)}&display=3&sort=sim"
        )
        headers = {
            "X-Naver-Client-Id": CLIENT_ID,
            "X-Naver-Client-Secret": CLIENT_SECRET,
        }

        res = requests.get(api_url, headers=headers, timeout=5)

        if res.status_code == 200:
            blog_items = res.json().get('items', [])

            if not blog_items:
                results.append({
                    '중분류명': cat,
                    '검색어': search_query,
                    '블로그제목': '결과없음',
                    '블로그링크': '',
                    '본문내용': '',
                })
            else:
                for blog in blog_items:
                    title = BeautifulSoup(blog['title'], 'html.parser').get_text()
                    link = blog['link']
                    content = get_blog_content(driver, link)
                    results.append({
                        '중분류명': cat,
                        '검색어': search_query,
                        '블로그제목': title,
                        '블로그링크': link,
                        '본문내용': content,
                    })
                    time.sleep(random.uniform(0.5, 1.0))  # 기존 1~1.5 → 0.5~1.0
        else:
            tqdm.write(f"⚠️ API 호출 실패 (Code: {res.status_code}) - {item}")

    except Exception as e:
        tqdm.write(f"⚠️ 예외 발생: {e} - {item}")

    finally:
        driver_pool.put(driver)  # 드라이버 반납 (필수)

    return results


# ─────────────────────────────────────────────
# 6. 중간 저장 함수
# ─────────────────────────────────────────────
def flush_to_csv(buffer: list):
    if not buffer:
        return
    df_buf = pd.DataFrame(buffer)
    file_exists = os.path.exists(OUTPUT_PATH)
    df_buf.to_csv(
        OUTPUT_PATH,
        mode='a',
        index=False,
        header=not file_exists,   # 파일 없으면 헤더 포함, 있으면 생략
        encoding='utf-8-sig'
    )
    buffer.clear()


# ─────────────────────────────────────────────
# 7. 메인
# ─────────────────────────────────────────────
def main():
    if not CLIENT_ID or not CLIENT_SECRET:
        print("오류: .env 파일에 네이버 API Key가 필요합니다.")
        return

    # ── 이어서 할 대상 추출 ───────────────────────────
    target_items = get_extra_target_items()
    if not target_items:
        print("더 이상 크롤링할 새로운 아이템이 없습니다.")
        return

    print(f"🚀 크롤링 시작: {len(target_items)}개 상품\n")

    # ── 드라이버 풀 생성 ─────────────────────────────
    driver_pool = create_driver_pool(DRIVER_COUNT)
    buffer = []  # 임시 저장 버퍼

    try:
        with ThreadPoolExecutor(max_workers=DRIVER_COUNT) as executor:
            futures = {
                executor.submit(process_item, driver_pool, cat, item): (cat, item)
                for cat, item in target_items
            }

            for i, future in enumerate(
                tqdm(as_completed(futures), total=len(futures),
                     desc="크롤링 진행률", ncols=100), 1
            ):
                try:
                    buffer.extend(future.result())
                except Exception as e:
                    tqdm.write(f"⚠️ future 처리 중 오류: {e}")

                # 50개마다 디스크에 저장 (중단돼도 유실 최소화)
                if i % 50 == 0:
                    flush_to_csv(buffer)
                    tqdm.write(f"💾 중간 저장 완료 ({i}개 처리됨)")

    except KeyboardInterrupt:
        # Ctrl+C 로 중단해도 지금까지 모은 데이터 저장
        tqdm.write("\n⚠️ 사용자 중단 감지 — 현재까지 수집 데이터 저장 중...")

    finally:
        # 드라이버 전부 종료
        while not driver_pool.empty():
            try:
                driver_pool.get_nowait().quit()
            except:
                pass

        # 버퍼 잔여분 저장
        flush_to_csv(buffer)
        print(f"\n🎉 완료! 결과 누적 저장 위치: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

수집 대상 아이템 분석 중...
MASTER 전체 유니크 상품명: 8118개
[success] 완료 처리된 항목: 3181개
[fail] 완료 처리된 항목: 8857개
총 완료(제외할) 항목 수: 4118개
남은 크롤링 대상: 4016개

🚀 크롤링 시작: 4016개 상품

드라이버 5개 초기화 중...
  드라이버 1/5 준비 완료
  드라이버 2/5 준비 완료
  드라이버 3/5 준비 완료
  드라이버 4/5 준비 완료
  드라이버 5/5 준비 완료


크롤링 진행률:   1%|▌                                           | 50/4016 [02:01<2:20:42,  2.13s/it]      

💾 중간 저장 완료 (50개 처리됨)


크롤링 진행률:   2%|█                                          | 100/4016 [04:32<2:12:38,  2.03s/it]      

💾 중간 저장 완료 (100개 처리됨)


...

💾 중간 저장 완료 (3950개 처리됨)


크롤링 진행률: 100%|█████████████████████████████████████████▊| 4000/4016 [2:48:26<00:29,  1.86s/it]      

💾 중간 저장 완료 (4000개 처리됨)


크롤링 진행률: 100%|██████████████████████████████████████████| 4016/4016 [2:49:03<00:00,  2.53s/it]



🎉 완료! 결과 누적 저장 위치: C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\블로그_나머지상품_크롤링.csv


In [2]:
df_rest = pd.read_csv(r'C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\블로그_나머지상품_크롤링.csv')
print(df_rest.shape)
df_rest.head()

(11011, 5)


,중분류명,검색어,블로그제목,블로그링크,본문내용
0,고급,세븐일레븐 G)크라시에유자셔벗140ml,세븐일레븐 아이스크림 크라시에 유자셔벗,https://blog.naver.com/osolo123/223862672512,좋은하루 보내셨나요?\n길을 걷다가 디저트가\n먹고 싶더라고요 ㅋ ㅋ\n날씨도 덥고...
1,가공우유,세븐일레븐 뉴초이스)아쌈밀크티300ml,[찐후기] ASSAM 아쌈 밀크티 세븐일레븐,https://blog.naver.com/namunorae/223285206929,안녕하세요\n오늘의 [ #찐후기 ]는\n#편의점 #음료수 찐후기입니다\n(모든 찐후...
2,가공우유,세븐일레븐 뉴초이스)아쌈밀크티300ml,세븐일레븐 아쌈 밀크티 이거 뭐지? 맛은 솔직히 말하면,https://blog.naver.com/memory_mj0329/224140125473,세븐일레븐에서 뭐 마실 거 없나 보다가\n괜히 밀크티가 땡겨서 하나 집어 들었다.\...
3,가공우유,세븐일레븐 뉴초이스)아쌈밀크티300ml,일본 세븐일레븐 편의점 홍차 밀크티 머신 자판기,https://blog.naver.com/areacorea/224105382748,일본 세븐일레븐 편의점 홍차 밀크티 머신 자판기\n글/ 나그네제이 사진/ 버선언니\...
4,국산맥주,세븐일레븐 오비)카스500ml캔,MLB 월드투어 서울시리즈 굿즈 오비맥주 카스 500ml 맥주잔...,https://blog.naver.com/oisuloi/223385089447,메이저리그 개막전 서울시리즈가\n이번주 핫한 키워드네요.\n전설적인 LA다저스 팀과...


In [3]:
df_rest.isnull().sum()

중분류명       0
검색어        0
블로그제목      0
블로그링크    380
본문내용     380
dtype: int64

In [7]:
df_rest[df_rest['본문내용'].isnull()]['검색어'].unique()

array(['세븐일레븐 롯데)자일리톨F24g', '세븐일레븐 동아)박카스F120ml',
       '세븐일레븐 훼밀리)복숭아무케루푸루푸루구미40g', '세븐일레븐 현대)미에로화이바350ml',
       '세븐일레븐 에프투)두레미니족발400g', '세븐일레븐 26년)얼음컵(벤티)', '세븐일레븐 아미)얼음컵(그란데)',
       '세븐일레븐 26년)얼음컵(그란데)', '세븐일레븐 제스)오레오바닐라바92ml', '세븐일레븐 오리온)초코송이4P',
       '세븐일레븐 MZ)호올스자몽27.9g', '세븐일레븐 해태)홈런볼128g', '세븐일레븐 썬푸드)오잉츄40g',
       '세븐일레븐 그래)여명808140ml', '세븐일레븐 현대)미에로화이바210ml', '세븐일레븐 유앤)쌀강정80g',
       '세븐일레븐 MQ)켈로그단백질바K50g', '세븐일레븐 현대)미에로화이바스파클링350ml',
       '세븐일레븐 빙그레)폴라포스포츠Zero120ml', '세븐일레븐 훼미리)아폴로35g',
       '세븐일레븐 MZ)호올스멘토립토스27.9g', '세븐일레븐 올리)순얼음3kg',
       '세븐일레븐 MDS)NEW투움바파스타230g', '세븐일레븐 쥬맥스)랭거스(망고)449ml',
       '세븐일레븐 예스)지렁이젤리샤우워', '세븐일레븐 직납)트롤리구미샤크100g', '세븐일레븐 쥬맥스)모구모구(멜론)',
       '세븐일레븐 유한)내일N100ml', '세븐일레븐 호연)꿀물180ml', '세븐일레븐 하겐)마카롱블루베리바',
       '세븐일레븐 일신)푸쉬팝', '세븐일레븐 매크로)파인아메캔디108.1g',
       '세븐일레븐 이노엔)티로그자두아이스티500ml', '세븐일레븐 현대)미에로화이바100ml',
       '세븐일레븐 MZ)밀카레오33g', '세븐일레븐 예스)트롤리XXL버거50g', '세븐일레븐 PB)참징어1마리60g',
       '세븐일레븐 일신)푸시팝플립앤딥25g', '세븐일레븐 엔제이)앙도넛100g',

In [9]:
import os
import pandas as pd

# ── 경로 설정 ──────────────────────────────────────────────────────────
MASTER_PATH  = r"C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\전처리_EDA\최종\pos_data_food_final_상품단위_filtered.csv"
SUCCESS_PATH = r"C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\seven_eleven_blog_reviews_success.csv"
FAIL_PATH    = r"C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\seven_eleven_blog_reviews_fail_v1.csv"
OUTPUT_PATH  = r"C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\블로그_나머지상품_크롤링.csv"
FINAL_PATH   = r"C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\블로그_전체상품_통합.csv"

COLS = ['중분류명', '검색어', '블로그제목', '블로그링크', '본문내용']

# ── 1. MASTER 로드 → 상품명:중분류명 매핑 ──────────────────────────────
df_master = pd.read_csv(MASTER_PATH, low_memory=False)
master_items = set(df_master['상품명'].dropna().str.strip().unique())

# 상품명 → 중분류명 매핑 딕셔너리
item_to_cat = (
    df_master.dropna(subset=['상품명', '상품중분류명'])
    .drop_duplicates(subset=['상품명'])
    .assign(상품명=lambda x: x['상품명'].str.strip())  # ← 체인 내부에서 strip
    .set_index('상품명')['상품중분류명']                 # ← 같은 DataFrame 참조
    .to_dict()
)
print(f"MASTER 전체 유니크 상품명: {len(master_items)}개")

# ── 2. OUTPUT 로드 → 이미 있는 상품명 파악 ───────────────────────────
df_output = pd.read_csv(OUTPUT_PATH, low_memory=False) if os.path.exists(OUTPUT_PATH) else pd.DataFrame(columns=COLS)

# OUTPUT의 검색어에서 상품명 추출
output_items = set(
    df_output['검색어']
    .dropna()
    .str.replace('세븐일레븐 ', '', n=1, regex=False)
    .str.strip()
)
print(f"OUTPUT에 이미 있는 상품명: {len(output_items)}개")

# ── 3. MASTER 기준으로 OUTPUT에 없는 상품 찾기 ───────────────────────
missing_items = master_items - output_items
print(f"OUTPUT에 없는 상품명: {len(missing_items)}개 → SUCCESS/FAIL에서 탐색")

# ── 4. SUCCESS / FAIL에서 누락 상품 데이터 수집 ──────────────────────
collected = []

for path, label in [(SUCCESS_PATH, 'success'), (FAIL_PATH, 'fail')]:
    if not os.path.exists(path):
        print(f"[{label}] 파일 없음, 스킵")
        continue

    df_src = pd.read_csv(path, low_memory=False)

    if '검색어' not in df_src.columns:
        print(f"[{label}] '검색어' 컬럼 없음, 스킵")
        continue

    # 검색어에서 상품명 추출 (prefix 제거)
    df_src['_상품명'] = (
        df_src['검색어']
        .dropna()
        .str.replace('세븐일레븐 ', '', n=1, regex=False)
        .str.strip()
    )

    # 누락 상품에 해당하는 행만 필터
    df_hit = df_src[df_src['_상품명'].isin(missing_items)].copy()
    print(f"[{label}] 누락 상품 중 {len(df_hit)}행 발견")

    if df_hit.empty:
        continue

    # 중분류명이 없으면 MASTER에서 채우기
    if '중분류명' not in df_hit.columns:
        df_hit['중분류명'] = df_hit['_상품명'].map(item_to_cat)
    else:
        mask = df_hit['중분류명'].isna()
        df_hit.loc[mask, '중분류명'] = df_hit.loc[mask, '_상품명'].map(item_to_cat)

    # 필요한 컬럼만 정리
    for col in COLS:
        if col not in df_hit.columns:
            df_hit[col] = ''

    collected.append(df_hit[COLS])

# ── 5. OUTPUT + 누락분 병합 후 저장 ──────────────────────────────────
if collected:
    df_extra = pd.concat(collected, ignore_index=True)
    print(f"\n추가할 행 수: {len(df_extra)}개")

    df_final = pd.concat([df_output[COLS], df_extra], ignore_index=True)
else:
    print("\n추가할 데이터 없음 (OUTPUT이 이미 완전함)")
    df_final = df_output[COLS].copy()

df_final.to_csv(FINAL_PATH, index=False, encoding='utf-8-sig')
print(f"\n✅ 통합 완료: {len(df_final)}행 → {FINAL_PATH}")

# ── 6. 커버리지 검증 ─────────────────────────────────────────────────
final_items = set(
    df_final['검색어']
    .dropna()
    .str.replace('세븐일레븐 ', '', n=1, regex=False)
    .str.strip()
)
still_missing = master_items - final_items
print(f"통합 후 여전히 누락된 상품: {len(still_missing)}개")
if still_missing:
    print("누락 상품 샘플:", list(still_missing)[:10])

MASTER 전체 유니크 상품명: 8118개
OUTPUT에 이미 있는 상품명: 4011개
OUTPUT에 없는 상품명: 4107개 → SUCCESS/FAIL에서 탐색
[success] 누락 상품 중 3148행 발견
[fail] 누락 상품 중 8857행 발견

추가할 행 수: 12005개

✅ 통합 완료: 23016행 → C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\블로그_전체상품_통합.csv
통합 후 여전히 누락된 상품: 0개


In [10]:
# df_final의 고유 상품 개수 (검색어에서 prefix 제거 후 기준)
unique_items_final = (
    df_final['검색어']
    .dropna()
    .str.replace('세븐일레븐 ', '', n=1, regex=False)
    .str.strip()
    .nunique()
)

print(f"df_final 고유 상품 수 : {unique_items_final}개")
print(f"MASTER  고유 상품 수  : {len(master_items)}개")
print(f"커버리지              : {unique_items_final / len(master_items) * 100:.1f}%")

df_final 고유 상품 수 : 8118개
MASTER  고유 상품 수  : 8118개
커버리지              : 100.0%


In [11]:
df_final.isnull().sum()

중분류명       0
검색어        0
블로그제목      0
블로그링크    380
본문내용     380
dtype: int64

In [13]:
df_final[df_final['본문내용'].isnull()]

,중분류명,검색어,블로그제목,블로그링크,본문내용
16,껌,세븐일레븐 롯데)자일리톨F24g,결과없음,NaN,NaN
143,기능성드링크,세븐일레븐 동아)박카스F120ml,결과없음,NaN,NaN
147,젤리류,세븐일레븐 훼밀리)복숭아무케루푸루푸루구미40g,결과없음,NaN,NaN
251,기능성드링크,세븐일레븐 현대)미에로화이바350ml,결과없음,NaN,NaN
360,냉장안주,세븐일레븐 에프투)두레미니족발400g,결과없음,NaN,NaN
...,...,...,...,...,...
10915,떡,세븐일레븐 제라헌스페셜세트,결과없음,NaN,NaN
10925,비스킷류,세븐일레븐 여리수)키도크래커샌드90g,결과없음,NaN,NaN
10963,노벨티,세븐일레븐 직납)천혜향바,결과없음,NaN,NaN
10967,와인,세븐일레븐 지비)탈마템프라니요750ml,결과없음,NaN,NaN
